# Knowledge Graph–based Recommendation (một dạng hybrid giữa Content-based + Graph-based + Metadata reasoning).
- **"Phim có diễn viên giống, genre giống, đạo diễn giống, kênh giống, hoặc nằm gần nhau 2–3 bước trong KG → embedding trở nên gần nhau."**

In [42]:
import pandas as pd
import numpy as np
import os
import json
import torch
import torch.nn as nn
import torch.optim as optim
from pathlib import Path
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from torch_geometric.nn import GATConv
from torch_geometric.utils import negative_sampling
from sklearn.model_selection import train_test_split as sk_train_test_split

"""
Cơ chế hoạt động:
1.  **Knowledge Graph Construction:** - Chuyển đổi metadata (program, actor, genre, director...) thành một đồ thị tri thức.
    - Tạo danh sách cạnh và ánh xạ thực thể.
2.  **Feature Engineering (SBERT):** - Sử dụng Sentence-BERT để mã hóa thông tin văn bản của program thành vector ngữ nghĩa.
    - Vector này được dùng làm đặc trưng khởi tạo cho node program trong đồ thị.
3.  **Graph Learning (KGAT):** - Huấn luyện mô hình GNN với cơ chế Attention (GAT) để học biểu diễn (embedding) cho từng node.
    - Mô hình học cách lan truyền thông tin từ actor/genre sang program.
4.  **Candidate Generation:** - Sử dụng embedding đã học để tìm kiếm lân cận.
    - Gợi ý các program có embedding gần nhất với rogram user vừa xem.
"""

METADATA_PATH = 'data/processed/program_metadata_full.parquet'
KG_DIR = 'data/kg'
SBERT_PATH = 'data/interim/sbert_embeddings.npy'
OUTPUT_PATH = "data/candidates/item_kg_emb.npy"
# tham số
EMBED_DIM = 128
EPOCHS = 200
LR = 0.01

os.makedirs(KG_DIR, exist_ok=True)
os.makedirs('data/interim', exist_ok=True)
os.makedirs('data/candidates', exist_ok=True)

# TẠO SBERT EMBEDDINGS
"""
Kiểm tra và tạo file SBERT Embeddings nếu chưa có.
- Input: Metadata (Title, Actor, Genre...).
- Process: Kết hợp văn bản -> Mã hóa bằng pre-trained SBERT model.
- Output: File .npy chứa ma trận vector đặc trưng.
"""
def ensure_sbert_exists():
    if os.path.exists(SBERT_PATH):
        return

    try:
        df = pd.read_parquet(METADATA_PATH).fillna('')
    except FileNotFoundError:
        return

    model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')

    # Kết hợp text để mã hóa
    df['text'] = df.apply(lambda x: f"{x['tv_show_title']} {x['actors']} {x['director']} {x['tv_show_genre_1']}", axis=1)
    embeddings = model.encode(df['text'].tolist(), show_progress_bar=True, batch_size=64)

    np.save(SBERT_PATH, embeddings)

# XÂY DỰNG ĐỒ THỊ TRI THỨC
"""
Xây dựng cấu trúc đồ thị từ metadata.
- Input: metadata dataframe.
- Process:
    - Tạo ID duy nhất cho mỗi thực thể.
    - Tạo danh sách cạnh kết nối program với các thực thể liên quan.
- Output:
    - kg_edge_index.pt: Tensor chứa danh sách cạnh.
    - kg_entity_map.json: Từ điển ánh xạ ID thực thể.
"""
def ensure_kg_exists():
    if os.path.exists(f"{KG_DIR}/kg_edge_index.pt"):
        return

    try:
        df_meta = pd.read_parquet(METADATA_PATH).fillna('')
    except: return

    entity_map = {} # node
    edges = [] # canh

    def get_id(name, prefix):
        clean = str(name).strip().lower()
        if not clean or clean == 'nan': return None
        key = f"{prefix}_{clean}"
        if key not in entity_map: entity_map[key] = len(entity_map)
        return entity_map[key]

    for _, row in tqdm(df_meta.iterrows(), total=len(df_meta), desc="Building KG"):
        prog_id = get_id(row['tv_show_id'], 'p')
        if prog_id is None: continue

        # Actor
        for actor in str(row.get('actors', '')).split(','):
            aid = get_id(actor, 'a')
            if aid: edges.extend([[prog_id, aid], [aid, prog_id]])
        # Genre
        for col in ['tv_show_genre_1', 'tv_show_genre_2']:
            gid = get_id(row.get(col, ''), 'g')
            if gid: edges.extend([[prog_id, gid], [gid, prog_id]])
        # Director & Channel
        did = get_id(row.get('director', ''), 'd')
        if did: edges.extend([[prog_id, did], [did, prog_id]])
        cid = get_id(row.get('channel_id', ''), 'c')
        if cid: edges.extend([[prog_id, cid], [cid, prog_id]])

    if not edges:
        return

    edge_index = torch.LongTensor(edges).t().contiguous()
    torch.save(edge_index, f"{KG_DIR}/kg_edge_index.pt")
    with open(f"{KG_DIR}/kg_entity_map.json", 'w') as f: json.dump(entity_map, f)

# MÔ HÌNH KGAT
def get_node_mappings(entity_map):
    p_keys = sorted([k for k in entity_map if k.startswith('p_')], key=lambda x: entity_map[x])
    return [entity_map[k] for k in p_keys], [k.split('_')[1] for k in p_keys]

class KGGNNEncoder(nn.Module):
    def __init__(self, num_ent, emb_dim, sbert, p_idx):
        super().__init__()
        self.emb = nn.Embedding(num_ent, emb_dim)   # Khoi tao embed co num_ent node va 1 embed co emb_dim chieu
        nn.init.xavier_uniform_(self.emb.weight.data)   # Khoi tao ngau nhien cho tung embed

        # Feature Fusion                   # khoi tao SBERT
        sbert = torch.tensor(sbert).float()
        if sbert.shape[1] != emb_dim:
            self.proj = nn.Linear(sbert.shape[1], emb_dim)
            sbert = self.proj(sbert)

        self.emb.weight.data[p_idx] = sbert.clone().detach()    # dung SBERT cho prgram
        self.emb.weight.requires_grad = True

        """
        Trong Transformer chuẩn (ví dụ GPT, BERT):
        → Attention được tính trên toàn bộ chuỗi (global).

        Nhưng trong Graph Attention Network (GAT):
        → Attention được tính chỉ giữa mỗi node và hàng xóm trực tiếp của nó.
        """ 
        self.conv1 = GATConv(emb_dim, emb_dim, heads=4, concat=False)    # Mang 2 tang
        self.conv2 = GATConv(emb_dim, emb_dim, heads=4, concat=False)

    def forward(self, edge_index):
        x = self.emb.weight
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index)
        return x

# HUẤN LUYỆN MÔ HÌNH
"""
Quy trình huấn luyện mô hìnhtrên KG.
- Load Data: đồ thị, map, SBERT.
- Split: chia tập cạnh train/test. 
- Loop: huấn luyện với positive (cạnh thật) và negative sampling (cạnh giả).
- Save: lưu embedding cuối cùng của các node rogram.
"""
def train_kgat():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # load Data
    edge_index = torch.load(f"{KG_DIR}/kg_edge_index.pt").to(device)
    with open(f"{KG_DIR}/kg_entity_map.json") as f: entity_map = json.load(f)
    sbert = np.load(SBERT_PATH)

    p_idx, p_ids = get_node_mappings(entity_map)
    sbert_p = sbert[:len(p_idx)] # Cắt đúng số lượng program

    model = KGGNNEncoder(len(entity_map), EMBED_DIM, sbert_p, p_idx).to(device)
    opt = optim.Adam(model.parameters(), lr=LR)
    crit = nn.BCEWithLogitsLoss()

    # chia cạnh
    all_idx = np.arange(edge_index.size(1))
    train_idx, _ = sk_train_test_split(all_idx, train_size=0.9, random_state=42)
    train_edges = edge_index[:, train_idx].to(device)

    # train
    print(f"   -> Training on {len(train_idx)} edges for {EPOCHS} epochs...")
    for ep in range(1, EPOCHS + 1):
        model.train(); opt.zero_grad()

        z = model(train_edges)
        # positive sampling
        pos = (z[train_edges[0]] * z[train_edges[1]]).sum(dim=1)
        # negative sampling
        neg_edges = negative_sampling(edge_index, num_nodes=len(entity_map), num_neg_samples=len(train_idx)).to(device)
        neg = (z[neg_edges[0]] * z[neg_edges[1]]).sum(dim=1)

        label = torch.cat([torch.ones_like(pos), torch.zeros_like(neg)])
        pred = torch.cat([pos, neg])

        loss = crit(pred, label)
        loss.backward()
        opt.step()

        if ep % 10 == 0: print(f"      Epoch {ep}: Loss {loss.item():.4f}")
    # SAVE
    model.eval()
    with torch.no_grad(): final_emb = model(edge_index).cpu().numpy()

    p_emb = final_emb[p_idx]
    np.save(OUTPUT_PATH, p_emb)

    with open(OUTPUT_PATH.replace('.npy', '.json'), 'w') as f:
        json.dump({pid: i for i, pid in enumerate(p_ids)}, f)

if __name__ == "__main__":
    ensure_sbert_exists()
    ensure_kg_exists()
    train_kgat()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/129 [00:00<?, ?it/s]

Building KG: 100%|██████████| 8234/8234 [00:00<00:00, 18054.31it/s]


   -> Training on 96474 edges for 200 epochs...
      Epoch 10: Loss 0.6178
      Epoch 20: Loss 0.5922
      Epoch 30: Loss 0.5696
      Epoch 40: Loss 0.5490
      Epoch 50: Loss 0.5282
      Epoch 60: Loss 0.5149
      Epoch 70: Loss 0.5009
      Epoch 80: Loss 0.5100
      Epoch 90: Loss 0.4894
      Epoch 100: Loss 0.4668
      Epoch 110: Loss 0.4511
      Epoch 120: Loss 0.4440
      Epoch 130: Loss 0.4387
      Epoch 140: Loss 0.4343
      Epoch 150: Loss 0.4385
      Epoch 160: Loss 0.4300
      Epoch 170: Loss 0.4227
      Epoch 180: Loss 0.4189
      Epoch 190: Loss 0.4152
      Epoch 200: Loss 0.4104


In [43]:
"""
Sử dụng Item Embeddings đã được huấn luyện từ mô hình GNN (KGAT) để tìm kiếm các chương trình tương đồng nhất cho mỗi người dùng.

Cơ chế hoạt động:
1.  **Data Loading:** Tải ma trận embedding (.npy) và bản đồ ánh xạ id (.json).
2.  **Normalization:** Chuẩn hóa L2 các vector embedding để phép nhân ma trận trở thành cosine similarity.
3.  **Seed Identification:** Xác định chương trình xem cuối cùng của mỗi user làm "hạt giống".
4.  **Nearest Neighbor Search:**
    - Với mỗi user, lấy vector của item hạt giống.
    - Tính độ tương đồng cosine với toàn bộ kho item.
    - Lấy Top-K item gần nhất làm ứng viên gợi ý.
5.  **Export:** Lưu danh sách ứng viên ra file Parquet.
"""
import os
import numpy as np
import pandas as pd
import json
import torch
from tqdm import tqdm

BASE_DIR = os.getcwd()
EMB_PATH = os.path.join(BASE_DIR, "data/candidates/item_kg_emb.npy")
MAP_PATH = os.path.join(BASE_DIR, "data/candidates/item_kg_emb.json")
TRAIN_PATH = os.path.join(BASE_DIR, "data/clean/train.parquet")
OUTPUT_PATH = os.path.join(BASE_DIR, "data/candidates/kg_embedding_candidates.parquet")

"""
Hàm chính thực thi quy trình tạo ứng viên từ KG Embeddings.
Bao gồm các bước kiểm tra file, load dữ liệu, tính toán tương đồng và lưu kết quả.
"""
def debug_and_run():

    files = {"Embedding": EMB_PATH, "Map JSON": MAP_PATH, "Train Data": TRAIN_PATH}
    missing = []
    for name, path in files.items():
        if os.path.exists(path):
            size = os.path.getsize(path) / 1024 # KB
        else:
            missing.append(name)

    if missing:
        return

    # load data
    try:
        # load NPY
        emb_matrix = np.load(EMB_PATH)

        # load json map
        with open(MAP_PATH, 'r') as f:
            id_map_raw = json.load(f)
        # chuẩn hóa Map
        id_map = {str(k).replace('.0', ''): v for k, v in id_map_raw.items()}
        index_to_id = {v: k for k, v in id_map.items()}

        # load train data
        df_train = pd.read_parquet(TRAIN_PATH)

    except Exception as e:
        return

    # tạo ứng viên

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tensor_emb = torch.tensor(emb_matrix, dtype=torch.float32).to(device)
    tensor_emb = torch.nn.functional.normalize(tensor_emb, p=2, dim=1)

    # Lấy item xem cuối cùng của mỗi user
    last_views = df_train.sort_values('start_time_view').groupby('user_id').last().reset_index()

    candidates = []
    TOP_K = 50
    BATCH_SIZE = 100

    user_batches = [last_views[i:i + BATCH_SIZE] for i in range(0, len(last_views), BATCH_SIZE)]

    # vòng lặp tìm
    for batch in tqdm(user_batches, desc="Searching"):
        seed_indices = []
        batch_users = []

        for _, row in batch.iterrows():
            raw_id = str(row['tv_show_id'])
            clean_id = raw_id.replace('.0', '')

            if clean_id in id_map:
                seed_indices.append(id_map[clean_id])
                batch_users.append(row['user_id'])

        if not seed_indices: continue

        # tính toán tương đồng
        seed_vecs = tensor_emb[seed_indices] # (Batch, Dim)
        scores = torch.matmul(seed_vecs, tensor_emb.t()) # (Batch, Num_Items)

        # lấy Top K
        top_vals, top_idxs = torch.topk(scores, k=TOP_K + 1, dim=1)

        # chuyển về CPU
        top_idxs = top_idxs.cpu().numpy()
        top_vals = top_vals.cpu().numpy()

        # lưu kết quả
        for i, user_id in enumerate(batch_users):
            seed_idx = seed_indices[i]
            rank = 1
            for j, item_idx in enumerate(top_idxs[i]):
                if item_idx == seed_idx: continue
                if rank > TOP_K: break

                rec_id_str = index_to_id[item_idx]

                candidates.append({
                    'user_id': user_id,
                    'tv_show_id': float(rec_id_str),
                    'score': float(top_vals[i][j]),
                    'rank': rank
                })
                rank += 1

    if candidates:
        df_cand = pd.DataFrame(candidates)
        os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
        df_cand.to_parquet(OUTPUT_PATH, index=False)
    else:
        print("\nKhông tìm thấy ứng viên nào.")

debug_and_run()

Searching: 100%|██████████| 49/49 [00:01<00:00, 47.98it/s]


content-based


In [44]:
import pandas as pd
import numpy as np
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from tqdm import tqdm
import os
from pathlib import Path

METADATA_PATH = "data/processed/program_metadata_full.parquet"
TRAIN_PATH = "data/clean/train.parquet"
OUTPUT_PATH = "data/candidates/content_profile.parquet"
TOP_K = 100
W_TITLE = 0.7
W_META = 0.3

def run_content_based_weighted():
    """
    Quy trình:
    1. Tải dữ liệu Metadata và Train Log.
    2. [Item Features] Tạo ma trận TF-IDF cho Title và Metadata.
    3. [User Profile] Tính toán vector đại diện cho User dựa trên lịch sử xem có trọng số.
    4. [Scoring] Tính điểm Cosine Similarity và trích xuất Top-K ứng viên.
    5. Lưu kết quả ra file Parquet.
    """
    # 1. LOAD DATA
    try:
        df_meta = pd.read_parquet(METADATA_PATH).fillna('')
        df_train = pd.read_parquet(TRAIN_PATH)
    except FileNotFoundError as e:
        return
    # 2. XÂY DỰNG ITEM FEATURES
    """
   Chuyển đổi thông tin văn bản của chương trình thành vector số học.
    - Xử lý Year Bucket.
    - Gom nhóm các thuộc tính phụ (Genre, Category...) thành chuỗi văn bản.
    - Vector hóa Title và Metadata string bằng TF-IDF.
    - Chuẩn hóa L2 để chuẩn bị cho tính Cosine Similarity.
    """
    # kiểm tra nếu có cột year
    if 'year_of_production' in df_meta.columns:
        def get_year_bucket(y):
            try:
                y = float(y)
                if y >= 2024: return 'year_new'
                if y >= 2020: return 'year_2020s'
                if y >= 2010: return 'year_2010s'
                return 'year_old'
            except: return 'year_unknown'
        df_meta['year_bucket'] = df_meta['year_of_production'].apply(get_year_bucket)
    else:
        df_meta['year_bucket'] = ''

    # danh sách các cột mong muốn
    wanted_cols = [
        'tv_show_category',
        'channel_title',
        'year_bucket',
        'tv_show_genre_1',
        'tv_show_genre_2',
        'tv_show_genre_3'
    ]

    # khởi tạo chuỗi rỗng
    df_meta['meta_str'] = ""

    for col in wanted_cols:
        if col in df_meta.columns:
            # dồn chuỗi
            df_meta['meta_str'] = df_meta['meta_str'] + " " + df_meta[col].astype(str).fillna('')
        else:
            print(f"Thiếu cột '{col}', bỏ qua.")

    df_meta['meta_str'] = df_meta['meta_str'].str.lower().str.strip()
    title_col = 'tv_show_title' if 'tv_show_title' in df_meta.columns else 'title'
    if title_col not in df_meta.columns:
        return

    tfidf_title = TfidfVectorizer(ngram_range=(1, 2), max_features=50000, min_df=2)
    item_vec_title = tfidf_title.fit_transform(df_meta[title_col].fillna(''))

    # vector hóa metadata
    tfidf_meta = TfidfVectorizer(max_features=5000, binary=True)
    item_vec_meta = tfidf_meta.fit_transform(df_meta['meta_str'])

    # chuẩn hóa L2
    item_vec_title = normalize(item_vec_title, axis=1)
    item_vec_meta = normalize(item_vec_meta, axis=1)

    # map id
    item_ids = df_meta['tv_show_id'].values
    item_id_to_idx = {idn: i for i, idn in enumerate(item_ids)}
    idx_to_item_id = {i: idn for i, idn in enumerate(item_ids)}

    # 3. XÂY DỰNG USER PROFILE
    """
    Tạo vector đại diện cho sở thích của người dùng.
    - Tính trọng số cho mỗi lượt xem (Duration * Time Decay).
    - Tạo ma trận tương tác (User x Item) với giá trị là trọng số đã tính.
    - Nhân ma trận tương tác với ma trận Item Features để ra User Vectors.
      (User_Vec = trung bình trọng số các Item_Vec mà user đã xem).
    """

    # tính weight
    max_date = df_train['start_time_view'].max()
    df_train['days_diff'] = (max_date - df_train['start_time_view']).dt.total_seconds() / (24*3600)
    df_train['time_weight'] = np.exp(-df_train['days_diff'] / 30.0)
    df_train['dur_weight'] = np.log1p(df_train['duration_view'])
    df_train['final_weight'] = df_train['time_weight'] * df_train['dur_weight']

    # map id sang index
    user_ids = df_train['user_id'].unique()
    user_id_to_idx = {uid: i for i, uid in enumerate(user_ids)}
    idx_to_user_id = {i: uid for i, uid in enumerate(user_ids)}

    df_train['item_idx'] = df_train['tv_show_id'].map(item_id_to_idx).fillna(-1).astype(int)
    df_train = df_train[df_train['item_idx'] != -1]
    df_train['user_idx'] = df_train['user_id'].map(user_id_to_idx)

    # ma trận tương tác
    interaction_matrix = sp.csr_matrix(
        (df_train['final_weight'], (df_train['user_idx'], df_train['item_idx'])),
        shape=(len(user_ids), len(item_ids))
    )
    interaction_matrix = normalize(interaction_matrix, norm='l1', axis=1)

    # tính user vectors
    user_vec_title = interaction_matrix.dot(item_vec_title)
    user_vec_meta = interaction_matrix.dot(item_vec_meta)

    # 4. TÍNH ĐIỂM VÀ LẤY CANDIDATES
    """
    Tìm kiếm và xếp hạng ứng viên.
    - Lặp qua từng batch người dùng để tiết kiệm bộ nhớ.
    - Tính Cosine Similarity giữa User Vector và Item Vector (Title & Meta).
    - Tổng hợp điểm: Score = 0.7 * Title_Sim + 0.3 * Meta_Sim.
    - Sử dụng `argpartition` để lấy nhanh Top-K item có điểm cao nhất.
    - Lưu kết quả vào danh sách candidates.
    """

    candidates = []
    batch_size = 1000
    num_users = user_vec_title.shape[0]

    for start_idx in tqdm(range(0, num_users, batch_size), desc="Scoring"):
        end_idx = min(start_idx + batch_size, num_users)

        u_title_batch = user_vec_title[start_idx:end_idx]
        u_meta_batch = user_vec_meta[start_idx:end_idx]

        # tính cosine
        sim_title = u_title_batch.dot(item_vec_title.T).toarray()
        sim_meta = u_meta_batch.dot(item_vec_meta.T).toarray()

        # weighted sum
        final_scores = (W_TITLE * sim_title) + (W_META * sim_meta)

        # lấy k phần tử lớn nhất
        if final_scores.shape[1] > TOP_K:
            top_k_part = np.argpartition(final_scores, -TOP_K, axis=1)[:, -TOP_K:]
        else:
            top_k_part = np.arange(final_scores.shape[1])[None, :]

        rows = np.arange(final_scores.shape[0])[:, None]
        top_k_scores = final_scores[rows, top_k_part]

        # sort lại Top-K để có thứ tự
        sort_indices = np.argsort(-top_k_scores, axis=1)

        for i in range(len(rows)):
            u_idx = start_idx + i
            real_user_id = idx_to_user_id[u_idx]

            sorted_indices = top_k_part[i][sort_indices[i]]
            sorted_scores = top_k_scores[i][sort_indices[i]]

            for rank, (item_idx, score) in enumerate(zip(sorted_indices, sorted_scores)):
                if score > 0:
                    candidates.append({
                        'user_id': real_user_id,
                        'tv_show_id': idx_to_item_id[item_idx],
                        'score': float(score),
                        'rank': rank + 1
                    })

    # 5. LƯU KẾT QUẢ
    if candidates:
        df_cand = pd.DataFrame(candidates)
        os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
        df_cand.to_parquet(OUTPUT_PATH, index=False)
    else:
        print("Không tìm thấy ứng viên nào.")

if __name__ == "__main__":
    run_content_based_weighted()

Thiếu cột 'tv_show_category', bỏ qua.


Scoring: 100%|██████████| 5/5 [00:03<00:00,  1.45it/s]
